# CS514 NMF Test B: Full-Matrix k-Sweep

This notebook summarizes the full-matrix NMF sweep. Unlike Test A, this test uses all 2,500 games rather than only the manually selected user-profile dimensions.

The goal is to ask whether the full ownership matrix naturally prefers exactly the 11 hand-labeled taste dimensions, or whether it also recovers temporal, bridge, franchise, and missing niche structures.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
BASE = ROOT / 'data' / 'processed' / 'cs514_network_analysis' / 'matrix_decomposition' / 'nmf_test_b'
FIG = ROOT / 'data' / 'processed' / 'cs514_network_analysis' / 'figures' / 'matrix_decomposition'

k_summary = pd.read_csv(BASE / 'nmf_test_b_k_summary.csv')
stability = pd.read_csv(BASE / 'nmf_test_b_seed_stability.csv')
canonical = pd.read_csv(BASE / 'nmf_test_b_canonical_seeds.csv')
top_games = pd.read_csv(BASE / 'nmf_test_b_top_games_by_component.csv')
type_share = pd.read_csv(BASE / 'nmf_test_b_component_type_mass_share.csv')
community_share = pd.read_csv(BASE / 'nmf_test_b_component_community_mass_share.csv')
dimension_similarity = pd.read_csv(BASE / 'nmf_test_b_component_included_dimension_similarity.csv')

k_summary

## Reconstruction Error and Stability

A useful k should lower reconstruction error while remaining stable across random seeds. The important pattern here is that k = 5, 8, 11, and 15 are perfectly stable across the tested seeds. k = 20 and k = 25 still reduce reconstruction error, but begin introducing unstable small components.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(k_summary['k'], k_summary['mean_reconstruction_error'], marker='o')
axes[0].set_title('NMF reconstruction error by k')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Mean reconstruction error')
axes[0].grid(alpha=0.25)

axes[1].plot(k_summary['k'], k_summary['mean_seed_stability'], marker='o', label='mean')
axes[1].plot(k_summary['k'], k_summary['min_seed_stability'], marker='o', label='minimum')
axes[1].axhline(0.90, color='gray', linestyle='--', linewidth=1)
axes[1].set_title('Component stability across seeds')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Matched cosine similarity')
axes[1].set_ylim(0, 1.05)
axes[1].legend()
axes[1].grid(alpha=0.25)

plt.tight_layout()

## Canonical Component Summaries

The script selected one canonical seed per k. The most useful inspection point is k = 15: it is more expressive than k = 11, still fully stable across seeds, and begins separating bridge, franchise, and temporal structures that were collapsed at lower k.

In [ ]:
def component_summary(k):
    seed = int(canonical.loc[canonical['k'] == k, 'canonical_seed'].iloc[0])
    type_part = type_share[(type_share['k'] == k) & (type_share['seed'] == seed)][
        ['nmf_component', 'best_community_type', 'best_community_type_share']
    ]
    comm_part = community_share[(community_share['k'] == k) & (community_share['seed'] == seed)][
        ['nmf_component', 'best_community', 'best_community_share']
    ]
    dim_part = dimension_similarity[(dimension_similarity['k'] == k) & (dimension_similarity['seed'] == seed)][
        ['nmf_component', 'best_included_dimension', 'best_included_dimension_cosine']
    ]
    return type_part.merge(comm_part, on='nmf_component').merge(dim_part, on='nmf_component')

component_summary(15)

In [ ]:
def top_component_games(k, n=12):
    seed = int(canonical.loc[canonical['k'] == k, 'canonical_seed'].iloc[0])
    rows = []
    subset = top_games[(top_games['k'] == k) & (top_games['seed'] == seed) & (top_games['component_game_rank'] <= n)]
    for component, group in subset.groupby('nmf_component'):
        rows.append({
            'component': component,
            'top_games': '; '.join(group['name'].astype(str).tolist()),
            'dominant_types': ', '.join(group['community_type'].astype(str).value_counts().head(3).index.tolist()),
        })
    return pd.DataFrame(rows)

top_component_games(15, n=12)

## Interpretation Notes

- k = 11 recovers most large taste communities, but compresses bridge/franchise/recent-release behavior into broad components.
- k = 15 is the strongest descriptive setting: it remains stable while separating new-hotness, gateway/bridge games, Unmatched/franchise ownership, Knizia/classic auction games, and heavy economic/Splotter-style games.
- k = 20 and k = 25 are useful for exploration, but the minimum seed stability drops sharply, so they should not be treated as canonical.
- The full-matrix result supports the Louvain taxonomy while also showing that the 11 user-profile dimensions are not exhaustive. They are a useful profile layer, not the complete latent structure of BGG ownership.